# Features extraction

In [58]:
# imports

from pathlib import Path # handling file paths

import numpy as np # numerical operations
import pandas as pd # CSV files

from scipy.signal import welch # spectral density estimation
from scipy.integrate import trapezoid # integration

import matplotlib.pyplot as plt # plotting

from tqdm.auto import tqdm # progress bars

from PIL import Image # display images in notebook

## 1. Settings

In [59]:
# Paths

OUTPUT_DIR = Path("outputs")
PREPROCESSED_DATA_PATH = OUTPUT_DIR / "preprocessing_iclabel"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("outputs") # directory for outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # create output directory if it doesn't exist
FIGURES_DIR = OUTPUT_DIR / "figures"
TABS_DIR = OUTPUT_DIR / "tabs"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PREPROCESSED_DATA_PATH / "adhdata_preprocessed_1_45Hz_ICA_ICLabel.csv" #"adhdata.csv"

print("Dataset exists:", DATA_PATH.exists())

Dataset exists: True


In [60]:
def save_figure(filename, dpi=300):
    """
    Save the current matplotlib figure inside outputs/figures.
    """
    output_path = FIGURES_DIR / filename
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
    print("Figure saved to:", output_path)

In [61]:
# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (929280, 21)


,Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,...,F8,T7,T8,P7,P8,Fz,Cz,Pz,Class,ID
0,-0.099400,-0.091206,0.001350,-0.004604,0.014640,0.011525,0.021660,0.019083,0.021076,0.023292,...,-0.013915,0.014726,0.009169,0.022103,0.013503,-0.001418,0.015728,0.018325,ADHD,v10p
1,-91.795794,-24.338983,4.265258,25.890910,16.008787,71.216396,-52.528348,155.637788,-344.061447,-96.506065,...,-147.009186,-39.906696,57.218303,3.049641,27.513166,9.076263,150.381776,276.303510,ADHD,v10p
2,-4.353990,-119.603606,88.367905,-87.421820,77.008811,56.286361,51.219698,126.535505,-84.033124,-25.407430,...,-196.138729,-41.646501,-40.761467,31.306048,-1.562695,45.153584,16.173137,263.448931,ADHD,v10p
3,-81.219945,-143.743866,-16.449875,-167.574185,52.284772,22.797771,27.256387,197.801757,-98.457920,66.685210,...,-241.684201,-47.414605,-75.146080,39.054968,50.530779,12.009090,174.691642,387.403406,ADHD,v10p
4,-63.619701,-115.721164,28.718512,-201.109154,109.516662,-41.479453,136.059633,250.871017,-203.553857,69.815218,...,-328.097697,-24.416166,-149.936028,87.684299,40.833137,-36.151109,114.512117,334.697045,ADHD,v10p


In [62]:
# EEG channels, frequency bands and regions

EEG_CHANNELS = [
    "Fp1", "Fp2", "F3", "F4",
    "C3", "C4",
    "P3", "P4",
    "O1", "O2",
    "F7", "F8",
    "T7", "T8",
    "P7", "P8",
    "Fz", "Cz", "Pz"
]

FS = 128  # sampling frequency in Hz

FREQ_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30)
}

REGIONS = {
    "frontal": ["Fp1", "Fp2", "F3", "F4", "F7", "F8", "Fz"],
    "central": ["C3", "C4", "Cz"],
    "parietal": ["P3", "P4", "P7", "P8", "Pz"],
    "temporal": ["T7", "T8"],
    "occipital": ["O1", "O2"]
}

## 2. Extracting Features

### Power Spectral Density estimation using Welch's method

EEG signals are time series: for each subject and each electrode, we have a sequence of voltage values changing over time. EEG is often analyzed in the frequency domain.

The goal of spectral analysis is to decompose a signal into sinusoidal components at different frequencies. Instead of asking only *how the signal changes over time*, we ask *how much power does the signal contain at each frequency*.

This is particularly useful for EEG because brain activity is often described in terms of frequency bands, such as delta, theta, alpha and beta.

### From time domain to frequency domain

Let $x[n]$ be a discrete EEG signal, where $n$ is the sample index. The signal is sampled at a fixed sampling frequency $f_s$, which in this dataset is: $f_s = 128 \text{ Hz}$

This means that 128 samples are recorded every second.

The Fourier transform allows us to represent the signal as a combination of sinusoidal waves with different frequencies. For a discrete signal, this is done using the Discrete Fourier Transform (DFT): $X[k] = \sum_{n=0}^{N-1} x[n] e^{-j 2\pi kn/N}$
where:
- $x[n]$ is the signal in the time domain;
- $X[k]$ is the frequency-domain representation;
- $N$ is the number of samples;
- $k$ indexes the frequency bins.

The squared magnitude of the Fourier coefficients gives information about the power of the signal at each frequency.
However, applying a single Fourier transform to the whole EEG signal can produce a noisy estimate, because EEG is non-stationary and contains random fluctuations.

### Power Spectral Density

The Power Spectral Density (PSD) describes how the power of a signal is distributed across frequencies. The unit of a PSD is typically $\frac{U^2}{Hz}$
where $U$ is the unit of the original signal, for example microvolts. This means that the PSD represents power per unit of frequency.

### Welch's method

In this project, the PSD is estimated using Welch's method. Welch's method is a standard approach for obtaining a more stable estimate of the PSD.
The main idea is:
1. split the signal into shorter segments;
2. apply a window function to each segment;
3. compute the Fourier transform of each windowed segment;
4. compute the periodogram of each segment;
5. average the periodograms across segments.

This reduces random fluctuations in the PSD estimate.
If the signal is divided into $K$ segments, Welch's PSD estimate can be written as:
$
\hat{S}_{xx}(f) = \frac{1}{K} \sum_{k=1}^{K} P_k(f)
$
where:
- $\hat{S}_{xx}(f)$ is the estimated PSD;
- $P_k(f)$ is the periodogram of segment $k$;
- $K$ is the number of segments.

A periodogram is approximately the squared magnitude of the Fourier transform of a segment:
$
P_k(f) \propto |X_k(f)|^2
$
where $X_k(f)$ is the Fourier transform of the $k$-th segment.

Averaging multiple periodograms produces a smoother and more reliable PSD estimate than using only one Fourier transform of the entire signal.

### Window length and overlap

In this notebook, Welch's method is applied with $n_{\text{perseg}} = 512$
Since the sampling frequency is $f_s = 128 \text{ Hz}$ each Welch window has duration of  $T = \frac{512}{128} = 4 \text{ seconds}$.

The frequency resolution is $\Delta f = \frac{f_s}{n_{\text{perseg}}}$, therefore
$\Delta f = \frac{128}{512} = 0.25 \text{ Hz}$.

A 50% overlap is used $n_{\text{overlap}} = 256$.

This means that consecutive windows share half of their samples. Overlap increases the number of segments used for averaging, improving the stability of the PSD estimate.

### EEG frequency bands

After estimating the PSD, band powers are computed by integrating the PSD within standard EEG frequency ranges.

The frequency bands used here are:

- $\delta: 1-4 \text{ Hz}$
- $\theta: 4-8 \text{ Hz}$
- $\alpha: 8-12 \text{ Hz}$
- $\beta: 12-30 \text{ Hz}$

For a given band $[f_1, f_2]$, the band power is computed as the area under the PSD curve in that frequency interval:
$P_{\text{band}} = \int_{f_1}^{f_2} PSD(f)\,df$

In practice, since the PSD is computed at discrete frequency bins, the integral is approximated numerically:

$P_{\text{band}} \approx \sum_{f=f_1}^{f_2} PSD(f)\Delta f$ or, more accurately, using trapezoidal integration.

For example, theta power is computed as $P_{\theta} = \int_{4}^{8} PSD(f)\,df$
and beta power as $P_{\beta} = \int_{12}^{30} PSD(f)\,df$


In [63]:
# Welch parameters

NPERSEG = 512      # 4 seconds at 128 Hz
NOVERLAP = 256     # 50% overlap

print("Window duration:", NPERSEG / FS, "seconds")
print("Frequency resolution:", FS / NPERSEG, "Hz")

Window duration: 4.0 seconds
Frequency resolution: 0.25 Hz


In [64]:
# functions for feature extraction

def compute_band_powers(signal, fs=128, nperseg=512, noverlap=256):
    """
    Compute absolute band powers from a 1D EEG signal using Welch PSD.

    Returns one value for each frequency band.
    """
    # compute PSD using Welch's method, returns frequencies and power spectral density values
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap
    ) 
    
    band_powers = {} #empty dictionary

    # loop through each frequency band
    for band_name, (fmin, fmax) in FREQ_BANDS.items(): 

        # select frequencies of the current band
        band_mask = (freqs >= fmin) & (freqs < fmax) # boolean mask

        # area under the PSD curve in this band (band power)
        # trapezoid method
        power = trapezoid(
            psd[band_mask],
            freqs[band_mask]
        )

        band_powers[band_name] = power # store result

    return band_powers



def extract_subject_features(subject_df, subject_id):
    """
    Extract EEG spectral features for a single subject.

    Output:
    one dictionary = one row in the final feature table.
    """
    features = {}

    subject_class = subject_df["Class"].iloc[0] # get class label

    features["ID"] = subject_id
    features["Class"] = subject_class
    features["n_samples"] = len(subject_df)
    features["duration_sec"] = len(subject_df) / FS # duration in seconds

    channel_band_powers = {}

    # channel-level features

    # loop through each EEG channel and compute band powers
    for channel in EEG_CHANNELS: 
        signal = subject_df[channel].values # 1D array of EEG values for this channel

        band_powers = compute_band_powers(
            signal,
            fs=FS,
            nperseg=NPERSEG,
            noverlap=NOVERLAP
        )

        channel_band_powers[channel] = band_powers

        # store band powers as features
        for band_name, power in band_powers.items():
            features[f"{channel}_{band_name}_power"] = power

        # compute theta/beta ratio for this channel
        theta = band_powers["theta"]
        beta = band_powers["beta"]

        # NaN if beta is zero
        features[f"{channel}_theta_beta_ratio"] = theta / beta if beta != 0 else np.nan

    # regional features
    for region_name, region_channels in REGIONS.items():
        for band_name in FREQ_BANDS.keys():
            values = [
                channel_band_powers[ch][band_name]
                for ch in region_channels
            ]

            # average across channel in this region
            features[f"{region_name}_{band_name}_power_mean"] = np.mean(values)

        region_theta = features[f"{region_name}_theta_power_mean"]
        region_beta = features[f"{region_name}_beta_power_mean"]

        features[f"{region_name}_theta_beta_ratio"] = (
            region_theta / region_beta if region_beta != 0 else np.nan
        )

    return features

In [65]:
## test on one subject and all channels

# test_id = df["ID"].iloc[0]
# test_channel = "Fz"

# test_signal = df.loc[df["ID"] == test_id, test_channel].values

# print("Test subject:", test_id)
# print("Signal length:", len(test_signal))
# print("Duration:", len(test_signal) / FS, "seconds")

# test_band_powers = compute_band_powers(
#     test_signal,
#     fs=FS,
#     nperseg=NPERSEG,
#     noverlap=NOVERLAP
# )

# test_band_powers

# test_subject_df = df[df["ID"] == test_id]

# test_features = extract_subject_features(
#     test_subject_df,
#     subject_id=test_id
# )

# print("Number of features:", len(test_features))
# test_features


# test_features = extract_subject_features(
#     test_subject_df,
#     subject_id=test_id
# )

# print("Number of features:", len(test_features))

# for key in list(test_features.keys())[:30]:
#     print(key, ":", test_features[key])

### Theta/Beta Ratio

In ADHD EEG research, the theta/beta ratio, $ TBR = \frac{P_{\theta}}{P_{\beta}}$, is often considered an important spectral feature.
In this project, theta/beta ratio is computed both for individual electrodes and for broader scalp regions, such as frontal, temporal, parietal and occipital regions.

### Feature extraction logic
The complete feature extraction pipeline is therefore:

$
\text{Raw EEG signal}
\rightarrow
\text{Welch PSD}
\rightarrow
\text{Band powers}
\rightarrow
\text{Theta/Beta ratio}
\rightarrow
\text{Subject-level feature table}
$

For each subject, the final output is one row containing spectral EEG features extracted from the full recording. 

In [66]:
all_features = []

subject_ids = df["ID"].unique()

for subject_id in tqdm(subject_ids):
    subject_df = df[df["ID"] == subject_id]

    subject_features = extract_subject_features(
        subject_df,
        subject_id=subject_id
    )

    all_features.append(subject_features)

features_df = pd.DataFrame(all_features)

print("Feature table shape:", features_df.shape)
display(features_df.head())

  0%|          | 0/121 [00:00<?, ?it/s]

Feature table shape: (121, 124)


,ID,Class,n_samples,duration_sec,Fp1_delta_power,Fp1_theta_power,Fp1_alpha_power,Fp1_beta_power,Fp1_theta_beta_ratio,Fp2_delta_power,...,temporal_delta_power_mean,temporal_theta_power_mean,temporal_alpha_power_mean,temporal_beta_power_mean,temporal_theta_beta_ratio,occipital_delta_power_mean,occipital_theta_power_mean,occipital_alpha_power_mean,occipital_beta_power_mean,occipital_theta_beta_ratio
0,v10p,ADHD,7680,60.0,1472.907558,669.659556,562.269713,533.361363,1.255546,1909.324181,...,4693.572037,1207.565470,1268.905357,1623.979998,0.743584,3698.215641,1822.605954,1012.215144,1522.134717,1.197401
1,v12p,ADHD,7680,60.0,7226.328256,2726.162552,980.979819,739.220613,3.687888,5349.759634,...,20712.200876,2257.740466,857.886192,663.568971,3.402420,10282.981955,3004.896965,1196.538210,1710.300841,1.756941
2,v14p,ADHD,7680,60.0,11175.990068,2243.042962,1056.760548,2750.672220,0.815453,13711.284587,...,19927.949014,3210.876579,1375.487313,2417.212101,1.328339,14610.563261,4446.438834,1999.308770,3305.138746,1.345311
3,v15p,ADHD,7680,60.0,6412.754290,959.282325,521.510255,604.024268,1.588152,8949.945201,...,8800.096107,1776.540277,1350.853340,1534.432993,1.157783,13620.186682,3002.769886,1347.582294,1517.707948,1.978490
4,v173,ADHD,7680,60.0,5075.802946,2158.031176,1485.302211,7310.184298,0.295209,4903.508953,...,1474.730803,774.847783,495.023407,1882.247319,0.411661,3670.139850,1872.638216,1024.738584,3009.926174,0.622154


In [67]:
# check

print(features_df["Class"].value_counts())
print("Missing values:", features_df.isna().sum().sum())
display(features_df.describe().T.head(20))

Class
ADHD       61
Control    60
Name: count, dtype: int64
Missing values: 0


,count,mean,std,min,25%,50%,75%,max
n_samples,121.0,7680.000000,0.000000,7680.000000,7680.000000,7680.000000,7680.000000,7680.000000
duration_sec,121.0,60.000000,0.000000,60.000000,60.000000,60.000000,60.000000,60.000000
Fp1_delta_power,121.0,9283.763667,9967.364944,863.021399,3414.678908,5196.844784,10170.698459,44294.089963
Fp1_theta_power,121.0,2376.160724,1922.367388,337.197806,1167.742902,1908.773683,2800.484404,9885.111133
Fp1_alpha_power,121.0,1028.172192,693.800403,96.962330,547.356713,902.161440,1318.214894,5379.346656
Fp1_beta_power,121.0,1804.929687,1477.690156,177.628392,944.524420,1467.880060,2090.773398,10520.121239
Fp1_theta_beta_ratio,121.0,1.524335,0.868219,0.266435,0.911612,1.239901,1.958566,4.510211
Fp2_delta_power,121.0,12543.272280,18426.018904,897.817890,3324.057502,6510.163097,13991.877475,155417.632765
Fp2_theta_power,121.0,2911.882647,3868.828886,336.688326,1171.410626,2009.251261,3265.520952,37246.483381
Fp2_alpha_power,121.0,1163.935361,1097.037551,260.484997,601.397529,875.143403,1275.107688,7988.543535


## 3. Saving new dataset

In [68]:
OUTPUT_PATH = OUTPUT_DIR / "eeg_features_data.csv"

features_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: outputs\eeg_features_data.csv
